# ETF AI ハンズオン
## Part 3: Cortex Search Service 構築

このノートブックでは、ETFファンド説明文書を使った **Cortex Search Service** を構築し、
セマンティック検索（意味検索）を体験します。

### このパートで体験できること

1. **ETFドキュメントの確認**: Cortex Search のデータソースを確認する
2. **テキストチャンキング**: ドキュメントを検索用に分割する
3. **Cortex Search Service の作成**: ETF情報の検索エンジンを構築する
4. **セマンティック検索のテスト**: 意味ベースの検索を体験する
5. **Agent 作成の準備**: Cortex Analyst + Cortex Search を組み合わせるための準備

### 🚀 体験ポイント

> **「キーワードではなく、"意味"で検索！」**
>
> 「リスクを抑えながら収益を得たい」と検索すると、
> 「インカム」「カバードコール」「高配当」のドキュメントがヒットします。

### 前提条件
- `setup.sql` と `part1_data_setup.ipynb`, `part2_cortex_analyst.ipynb` の実行完了

> ⏱️ **このパートの目安時間: 25分**

In [ ]:
USE DATABASE ETF_AI_HANDSON_DB;
USE SCHEMA DEMO_SCHEMA;
USE WAREHOUSE DEMO_WH;
SELECT CURRENT_DATABASE() AS DB, CURRENT_SCHEMA() AS SCHEMA, CURRENT_WAREHOUSE() AS WH;

## 1. ETFドキュメントの確認

Cortex Search のデータソースとなる `ETF_DESCRIPTIONS` テーブルを確認します。

> ⏱️ **目安: 5分**

In [ ]:
-- ETFドキュメント一覧（Cortex Search の入力データ）
SELECT
    DOC_ID          AS "文書ID",
    ETF_CODE        AS "コード",
    ETF_NAME        AS "ETF名称",
    SECTION         AS "セクション",
    DOC_TYPE        AS "種別",
    LEN(CONTENT)    AS "文字数"
FROM ETF_DESCRIPTIONS
ORDER BY ETF_CODE, SECTION;

In [ ]:
-- 2641 グローバルリーダーズのドキュメントを確認
SELECT SECTION, CONTENT
FROM ETF_DESCRIPTIONS
WHERE ETF_CODE = '2641'
ORDER BY DOC_ID;

## 2. マーケットニュースを Cortex Search に追加

ETFドキュメントに加えて、マーケットニュースも検索できるようにします。
検索用の統合テーブルを作成します。

> ⏱️ **目安: 5分**

In [ ]:
-- Cortex Search 用の統合ビューを作成（ETFドキュメント + マーケットニュース）
CREATE OR REPLACE VIEW ETF_AI_HANDSON_DB.AI.ETF_KNOWLEDGE_BASE_V AS
    -- ETFファンド説明文書
    SELECT
        DOC_ID          AS CHUNK_ID,
        ETF_CODE        AS ETF_CODE,
        ETF_NAME        AS ETF_NAME,
        SECTION         AS SECTION,
        DOC_TYPE        AS SOURCE_TYPE,
        PUBLISHED_DATE  AS PUBLISHED_DATE,
        CONTENT         AS CHUNK_TEXT
    FROM ETF_DESCRIPTIONS

    UNION ALL

    -- マーケットニュース
    SELECT
        NEWS_ID                AS CHUNK_ID,
        RELEVANT_ETFS          AS ETF_CODE,
        NULL                   AS ETF_NAME,
        CATEGORY               AS SECTION,
        'マーケットニュース'    AS SOURCE_TYPE,
        NEWS_DATE              AS PUBLISHED_DATE,
        HEADLINE || '\n' || BODY AS CHUNK_TEXT
    FROM MARKET_NEWS;

-- 確認
SELECT SOURCE_TYPE, COUNT(*) AS "件数"
FROM ETF_AI_HANDSON_DB.AI.ETF_KNOWLEDGE_BASE_V
GROUP BY SOURCE_TYPE;

## 3. Cortex Search Service の作成

`CREATE CORTEX SEARCH SERVICE` で ETFナレッジベースの検索エンジンを構築します。

**ポイント**: `ON CHUNK_TEXT` で検索対象カラムを、`ATTRIBUTES` でフィルタリング用カラムを指定します。

> ⏱️ **目安: 10分**（インデックス構築に数分かかります）

In [ ]:
-- Cortex Search Service を作成（COMPUTE_WH でインデックスを構築）
USE WAREHOUSE COMPUTE_WH;

CREATE OR REPLACE CORTEX SEARCH SERVICE ETF_AI_HANDSON_DB.AI.ETF_KNOWLEDGE_SEARCH
    ON CHUNK_TEXT
    ATTRIBUTES ETF_CODE, SOURCE_TYPE, SECTION, PUBLISHED_DATE
    WAREHOUSE = COMPUTE_WH
    TARGET_LAG = '1 day'
    COMMENT = 'ETFファンドドキュメント＆マーケットニュースの統合検索サービス'
    AS (
        SELECT
            CHUNK_ID,
            ETF_CODE,
            ETF_NAME,
            SECTION,
            SOURCE_TYPE,
            PUBLISHED_DATE,
            CHUNK_TEXT
        FROM ETF_AI_HANDSON_DB.AI.ETF_KNOWLEDGE_BASE_V
    );

SELECT 'Cortex Search Service 作成開始！インデックス構築に数分かかります...' AS STATUS;

In [ ]:
-- Search Service の状態確認
SHOW CORTEX SEARCH SERVICES IN SCHEMA ETF_AI_HANDSON_DB.AI;

## 4. セマンティック検索のテスト

Cortex Search Service に Python から問い合わせてみます。

> ⏱️ **目安: 10分**

In [ ]:
from snowflake.snowpark.context import get_active_session
import json

session = get_active_session()

def search_etf_knowledge(query: str, limit: int = 5, filter_dict: dict = None):
    """ETFナレッジベースをセマンティック検索する"""
    from snowflake.core import Root
    root = Root(session)

    search_service = (
        root
        .databases["ETF_AI_HANDSON_DB"]
        .schemas["AI"]
        .cortex_search_services["ETF_KNOWLEDGE_SEARCH"]
    )

    search_params = {
        "query": query,
        "columns": ["CHUNK_ID", "ETF_CODE", "ETF_NAME", "SECTION", "SOURCE_TYPE", "CHUNK_TEXT"],
        "limit": limit
    }
    if filter_dict:
        search_params["filter"] = filter_dict

    resp = search_service.search(**search_params)
    return resp.results

def display_search_results(query: str, results):
    print(f"検索クエリ: 「{query}」")
    print(f"ヒット件数: {len(results)} 件")
    print("=" * 70)
    for i, r in enumerate(results, 1):
        print(f"\n[{i}] {r.get('SOURCE_TYPE', '')} | ETF: {r.get('ETF_CODE', '')} | {r.get('SECTION', '')}")
        print(r.get('CHUNK_TEXT', '')[:200] + "...")

print("Cortex Search クライアント準備完了")

In [ ]:
# Q1: キーワード検索 vs セマンティック検索の違いを体感
# 「インカム」というキーワードは含まないが、意味的に関連するドキュメントがヒットするか？
query = "リスクを抑えながら安定的に収益を得たい"
results = search_etf_knowledge(query, limit=5)
display_search_results(query, results)

In [ ]:
# Q2: 半導体・AI関連の最新ニュースを検索
query = "AI需要と半導体サプライチェーンへの影響"
results = search_etf_knowledge(query, limit=5)
display_search_results(query, results)

In [ ]:
# Q3: 特定ETFのリスク情報を検索（フィルタリングあり）
query = "元本割れリスクと価格変動"
results = search_etf_knowledge(
    query,
    limit=5,
    filter_dict={"@eq": {"SOURCE_TYPE": "目論見書"}}
)
display_search_results(query, results)

In [ ]:
# Q4: 日銀利上げのETFへの影響（ニュースから）
query = "日本銀行の金融政策とETFへの影響"
results = search_etf_knowledge(query, limit=5)
display_search_results(query, results)

In [ ]:
-- SQL経由でも Cortex Search を呼び出せる
USE WAREHOUSE DEMO_WH;

SELECT PARSE_JSON(
    SNOWFLAKE.CORTEX.SEARCH_PREVIEW(
        'ETF_AI_HANDSON_DB.AI.ETF_KNOWLEDGE_SEARCH',
        '{
            "query": "高配当ETFの分配金について",
            "columns": ["CHUNK_ID", "ETF_CODE", "SECTION", "CHUNK_TEXT"],
            "limit": 3
        }'
    )
) AS 検索結果;

## 5. Cortex Agent の作成準備（マニュアル操作）

これで Cortex Analyst (Semantic View) と Cortex Search (Search Service) の両方が揃いました。
次のステップとして、これらを組み合わせた **Cortex Agent** を Snowflake Intelligence 経由で設定します。

### Agent 作成の手順（UI操作）

1. **Snowsight** にログイン
2. 左メニュー「**AI & ML**」→「**Agents**」をクリック
3. 「**+ New Agent**」をクリック
4. 以下を設定：

| 設定項目 | 値 |
|---|---|
| **Agent 名** | `ETF_PORTFOLIO_COPILOT` |
| **説明** | ETFポートフォリオ分析AIアシスタント |
| **Semantic View (Cortex Analyst)** | `ETF_AI_HANDSON_DB.AI.PORTFOLIO_ANALYTICS_VIEW` |
| **Search Service** | `ETF_AI_HANDSON_DB.AI.ETF_KNOWLEDGE_SEARCH` |
| **System Prompt** | 下記参照 |

### システムプロンプト（コピーして使用）

```
あなたは ポートフォリオマネージャーをサポートする
AI アシスタント（Portfolio Copilot）です。

## 役割
- ポートフォリオのパフォーマンス分析・リスク評価をサポートする
- ETFファンドの情報を検索し、投資判断に役立つ情報を提供する
- マーケットニュースを分析し、ポートフォリオへの影響を評価する

## 使用ツール
- Cortex Analyst (PORTFOLIO_ANALYTICS_VIEW): ポートフォリオ定量データの分析
- Cortex Search (ETF_KNOWLEDGE_SEARCH): ETFファンド情報とニュースの検索

## 回答スタイル
- 日本語で簡潔かつ正確に回答する
- 定量的なデータ（数値・比率）を活用して具体的に答える
- 投資判断はあくまでサポートであり、最終判断はPMが行うことを明記する

## 注意事項
- 回答は参考情報であり、投資助言ではありません
- データは2026年4月時点のものです
```

## 6. Snowflake Intelligence デモ質問集

Agent の設定が完了したら、以下の 3 シナリオで **Snowflake Intelligence** を体験してください。  
それぞれ **1〜2 文の自然な質問** で、Cortex Analyst（定量）と Cortex Search（定性）が自動的に連携します。

---

### シナリオ 1: 朝のポートフォリオブリーフィング ☀️

> **状況**: 週次レビュー前に、担当ポートフォリオの現状を素早く把握したい

```
グローバル分散成長ポートフォリオ（P004）の直近3ヶ月のパフォーマンスと
主要保有ETFの状況を教えてください。
また、このポートフォリオに影響しそうな最近のニュースがあれば合わせて確認してください。
```

**期待される動作**:
- Cortex Analyst → `FACT_PERFORMANCE` から月次リターンを集計
- Cortex Search → `MARKET_NEWS` から関連ニュースをセマンティック検索
- 定量データ + 定性情報を 1 回答にまとめて提示

---

### シナリオ 2: セクターイベントの影響確認 ⚡

> **状況**: 半導体セクターに関するニュースが出た。保有ポートフォリオへの影響を素早く把握したい

```
半導体関連のETFを保有しているポートフォリオはどれですか？
それぞれの組入比率と直近の損益を教えてください。
また、半導体セクターの最近の動向も合わせて教えてください。
```

**期待される動作**:
- Cortex Analyst → `FACT_HOLDINGS` から半導体ETF（2644）の保有状況を抽出
- Cortex Analyst → `FACT_PERFORMANCE` から直近損益を算出
- Cortex Search → 半導体関連ニュース・ETFドキュメントを検索

> 💡 **ポイント**: 9つのタスクを一度に投げず、「影響確認 → 詳細深掘り」と
> **会話を積み重ねる**ことで、Agentの真価が伝わります

---

### シナリオ 3: ETF比較・選定サポート 📊

> **状況**: インカム強化のためにETFを追加したい。特徴を比較して選びたい

```
インカム重視でポートフォリオを強化したいと考えています。
配当利回りが高いETFをリストアップして、それぞれの特徴と
リスク・リターン特性を比較してください。
```

**期待される動作**:
- Cortex Analyst → `DIM_ETF` からインカム系ETFを抽出・比較
- Cortex Search → 各ETFのファンド説明・特性情報を検索
- 比較表形式での提示

---

> ⚡ **Agentic AI のポイント**:
> - **シナリオ 1**: ルーティン分析を秒速で実行
> - **シナリオ 2**: イベント発生時の即時影響把握（複雑な9ステップ分析をシンプルな会話に）
> - **シナリオ 3**: 意思決定前のリサーチを自動化
>
> 定量データ（Cortex Analyst）× 定性情報（Cortex Search）の組み合わせが、
> ポートフォリオマネージャーの判断スピードを劇的に向上させます。

## まとめ

Part 3 完了！ETFナレッジベースの Cortex Search Service が構築されました。

### 作成したオブジェクト

| オブジェクト | 種別 | 場所 |
|---|---|---|
| `ETF_KNOWLEDGE_BASE_V` | View | `ETF_AI_HANDSON_DB.AI` |
| `ETF_KNOWLEDGE_SEARCH` | Cortex Search Service | `ETF_AI_HANDSON_DB.AI` |

### ハンズオン全体のまとめ

```
setup.sql
    ↓ データ基盤構築
part1: AI Functions
    ↓ ニュース要約・センチメント・分類・抽出
part2: Cortex Analyst (PORTFOLIO_ANALYTICS_VIEW)
    ↓ 自然言語でポートフォリオ定量分析
part3: Cortex Search (ETF_KNOWLEDGE_SEARCH)
    ↓ ETFドキュメント＆ニュースのセマンティック検索
[手動操作] Cortex Agent 作成
    ↓ Analyst + Search の統合
Snowflake Intelligence
    └─ ETF_PORTFOLIO_COPILOT
```

### お疲れ様でした！

ポートフォリオマネージャー向け AI アシスタントの基盤が完成しました。
今後の拡張として以下が考えられます：

- **実データ連携**: Bloomberg / Refinitiv などの市場データを追加
- **PDF ファクトシート**: `AI_PARSE_DOCUMENT` で ETF運用会社 の月次レポートを自動取込
- **アラート機能**: 閾値超過時に Slack/メール通知（Tasks + External Functions）
- **バックテスト**: 過去データを使ったポートフォリオ最適化シミュレーション